**UE 3.2 - TP n°3 - Prédiction de la dose à l'utérus lors d'une exposition CT**

**ETAPE 1  - INITIALISATION ET IMPORTATION DES DONNEES**

In [2]:
#Import standard libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys
import seaborn as sns
from IPython.display import Markdown
from importlib import reload
import scipy.stats as stats
from scipy.optimize import curve_fit
from scipy.stats import pearsonr, spearmanr

#Import SckitLearn tools
from sklearn.model_selection import KFold, cross_val_score, train_test_split, GridSearchCV, StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.svm import SVR
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import BaseEstimator, RegressorMixin, clone

#Import dataset
data = pd.read_excel("database_dose_uterus.xlsx", sheet_name = 0, header=0)
df = pd.DataFrame(data)

#Define input features and output features
data = data.sample(frac=1., axis=0)

#Input features
x_data = data.drop(['N_data', 'CTDIvol_mGy', 'PDL_mGycm', 'Duterus_mSv'], axis=1)

#Output features
y_data1 = data[['CTDIvol_mGy']]
y_data2 = data[['Duterus_mSv']]

#Split data into Train and Test sets (for both models)
np.random.seed(42)

#Train-test split for CTDIvol model
x_train1, x_test1, y_train1, y_test1, train_exp1, test_exp1 = train_test_split(x_data, y_data1, data['N_data'], test_size = 0.2, random_state=42, stratify=x_data['U_kV'])

#Train-test split for Duterus model
x_train2, x_test2, y_train2, y_test2, train_exp2, test_exp2 = train_test_split(x_data, y_data2, data['N_data'], test_size = 0.2, random_state=42, stratify=x_data['U_kV'])

#Convert y_train into 1D array
y_train1 = np.ravel(y_train1)
y_train2 = np.ravel(y_train2)

#Convert Pandas series into Numpy arrays
x_train1 = np.array(x_train1)
y_train1 = np.array(y_train1)
x_test1 = np.array(x_test1)
y_test1 = np.array(y_test1)
x_train2 = np.array(x_train2)
y_train2 = np.array(y_train2)
x_test2 = np.array(x_test2)
y_test2 = np.array(y_test2)

#Define best single model for CTDIvol prediction
best_model1 = SVR(
    C=100,
    kernel='rbf',
    gamma='scale',
    epsilon=0.2,
    verbose=0
)

#Fit the model to the train subset
best_model1.fit(x_train1, y_train1)

#Define best single model for Duterus prediction
best_model2 = SVR(
    C=100,
    kernel='rbf',
    gamma='scale',
    epsilon=0.1,
    verbose=0
)

#Fit the model to train subset
best_model2.fit(x_train2, y_train2)

#Final single model for CTDIvol prediction

scaler_x_final1 = StandardScaler()
scaler_y_final1 = StandardScaler()

x_train_scaled1 = scaler_x_final1.fit_transform(x_train1)
y_train_scaled1 = scaler_y_final1.fit_transform(y_train1.reshape(-1,1)).flatten()

#Fit final single model
final_model1 = best_model1
final_model1.fit(x_train_scaled1, y_train_scaled1)

y_train_pred_scaled_final1 = final_model1.predict(x_train_scaled1)
y_train_pred_final1 = scaler_y_final1.inverse_transform(y_train_pred_scaled_final1.reshape(-1,1)).flatten()

final_mse1 = mean_squared_error(y_train1, y_train_pred_final1)

#Final single model on Duterus prediction

scaler_x_final2 = StandardScaler()
scaler_y_final2 = StandardScaler()

x_train_scaled2 = scaler_x_final2.fit_transform(x_train2)
y_train_scaled2 = scaler_y_final2.fit_transform(y_train2.reshape(-1,1)).flatten()

#Fit final single model
final_model2 = best_model2
final_model2.fit(x_train_scaled2, y_train_scaled2)

y_train_pred_scaled_final2 = final_model2.predict(x_train_scaled2)
y_train_pred_final2 = scaler_y_final2.inverse_transform(y_train_pred_scaled_final2.reshape(-1,1)).flatten()

final_mse2 = mean_squared_error(y_train2, y_train_pred_final2)





**ETAPE 2 - INTERFACE UTILISATEUR**

In [11]:
#Input dialog boxes
def get_input_features():
  features = []
  print("Please enter the values for the input features:")
  for name in feature_names_inbox:
    feature_value = float(input(f"{name}: "))
    features.append(feature_value)
  return features

#List of input features
feature_names_inbox = [
    "Tension du tube RX (kV)",
    "Charge délivrée (mA.s)",
    "Largeur totale de collimation (mm)",
    "Déplacement de la table (mm)",
    "Pitch",
    "Epaisseur de coupe reconstruite (mm)"
    ]

input_features = get_input_features()


#Whole set data recuperation and standardization
mean = x_data.mean()
std = x_data.std()
x_data_standardized = (x_data - mean) / std

#Convert y_data into a Numpy array
y_data1 = np.ravel(y_data1)
y_data2 = np.ravel(y_data2)

#Standardization of user inputs
means = np.mean(x_data, axis=0)
stds = np.std(x_data, axis=0)

manually_standardized_input_features = [(x - mean) / std for x, mean, std in zip (input_features, means, stds)]

#Fit models on whole dataset
regression_model1 = final_model1.fit(x_data_standardized, y_data1)
regression_model2 = final_model2.fit(x_data_standardized, y_data2)

#Make a prediction
predicted_CTDIvol = regression_model1.predict([manually_standardized_input_features])
predicted_Duterus = regression_model2.predict([manually_standardized_input_features])

#Outputs recuperation (CTDIvol and Duterus)
CTDIvol_output = predicted_CTDIvol[0]
Duterus_output = predicted_Duterus[0]

#Display results of the predictions
final_data = [CTDIvol_output, Duterus_output]
final_labels = ['CTDIvol (mGy)', 'Duterus (mSv)']
decimal_places = [2, 2]

formatted_data = [f"{value:.{dp}f}" for value, dp in zip(final_data, decimal_places)]

df = pd.DataFrame({'Quantity': final_labels, 'Value': formatted_data})

df_styled = df.style.set_properties(**{'text-align': 'center'})
df_styled


Please enter the values for the input features:
Tension du tube RX (kV): 100
Charge délivrée (mA.s): 33
Largeur totale de collimation (mm): 19.2
Déplacement de la table (mm): 19.2
Pitch: 1
Epaisseur de coupe reconstruite (mm): 2


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SVR was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SVR was fitted with feature names
  warnings.warn(


,Quantity,Value
0,CTDIvol (mGy),1.72
1,Duterus (mSv),2.40
